# Phase 4 — Add Knowledge & Retrieval (RAG)
**Goal:** Index product documentation, implement semantic search with **Pinecone**, and connect retrieval results to the LLM.

---

In [ ]:
import os
import sys
sys.path.insert(0, os.path.abspath(".."))  # allow importing agent/

from dotenv import load_dotenv
load_dotenv("../.env")

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document
from pinecone import Pinecone
from agent.retriever import build_vectorstore, get_retriever, DOCS_PATH

llm = ChatOpenAI(model=os.environ.get("OPENAI_MODEL", "gpt-4o"), temperature=0)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
print("Setup complete.")

Exception: The official Pinecone python package has been renamed from `pinecone-client` to `pinecone`. Please remove `pinecone-client` from your project dependencies and add `pinecone` instead. See the README at https://github.com/pinecone-io/pinecone-python-client for more information on using the python SDK.

## 4.1 Load and Inspect Documents

In [2]:
loader = DirectoryLoader(DOCS_PATH, glob="*.txt", loader_cls=TextLoader, show_progress=True)
docs = loader.load()

print(f"Loaded {len(docs)} documents:")
for doc in docs:
    source = doc.metadata.get("source", "unknown")
    print(f"  {os.path.basename(source):30s}  {len(doc.page_content):>6} chars")

  0%|          | 0/6 [00:00<?, ?it/s]

100%|██████████| 6/6 [00:00<00:00, 34.21it/s]

Loaded 6 documents:
  billing_policy.txt                3549 chars
  features.txt                      2507 chars
  getting_started.txt               2041 chars
  integrations.txt                  3388 chars
  known_issues.txt                  2660 chars
  troubleshooting.txt               3669 chars


## 4.2 Chunk Documents

In [3]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " ", ""],
)
chunks = splitter.split_documents(docs)

print(f"Created {len(chunks)} chunks (avg {sum(len(c.page_content) for c in chunks)//len(chunks)} chars each)")
print("\nSample chunk:")
print(chunks[5].page_content)
print("\nSource:", chunks[5].metadata.get("source"))

Created 52 chunks (avg 344 chars each)

Sample chunk:
## Refund Policy
- Monthly subscriptions: No refunds are issued for partial months already paid.
- Annual subscriptions: A pro-rated refund is available within the first 30 days of the annual subscription start date. After 30 days, no refunds are issued for annual subscriptions.
- Free trial: If you cancel during a free trial period, no charges apply.

Source: c:\Users\admin\AgenticAI_Course\Capstone project\data\product_docs\billing_policy.txt


## 4.3 Embed and Store in Pinecone
⚠️ This cell calls the OpenAI Embeddings API and upserts to Pinecone. Safe to re-run — skips upsert if vectors already exist.

In [ ]:
# build_vectorstore() skips upserting if the index already has vectors.
# Pass force_rebuild=True to re-index after document changes.
vectorstore = build_vectorstore(force_rebuild=False)

# Show Pinecone index stats
PINECONE_INDEX_NAME = os.environ.get("PINECONE_INDEX_NAME", "taskflow-kb")
pc = Pinecone(api_key=os.environ.get("PINECONE_API_KEY", ""))
index = pc.Index(PINECONE_INDEX_NAME)
stats = index.describe_index_stats()
print(f"Pinecone index : {PINECONE_INDEX_NAME}")
print(f"Total vectors  : {stats.get('total_vector_count', 0)}")
print(f"Dimension      : {stats.get('dimension', 'N/A')}")

Pinecone index 'taskflow-kb' already exists.
Loading documents from 'c:\Users\admin\AgenticAI_Course\Capstone project\data/product_docs' ...


100%|██████████| 6/6 [00:00<00:00, 1000.47it/s]

Loaded 6 documents → 52 chunks.
Upserting chunks to Pinecone ...


Upserted 52 chunks to index 'taskflow-kb'.
Pinecone index : taskflow-kb
Total vectors  : 52
Dimension      : 1536


## 4.4 Test Semantic Retrieval

In [5]:
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 3})

test_queries = [
    "How do I add a sub-task?",
    "What is the refund policy for annual subscriptions?",
    "Why is my Gantt chart loading slowly?",
    "How do I connect TaskFlow to Slack?",
]

for q in test_queries:
    docs_found = retriever.invoke(q)
    print(f"QUERY: {q}")
    for i, d in enumerate(docs_found, 1):
        src = os.path.basename(d.metadata.get("source", "unknown"))
        print(f"  [{i}] {src}: {d.page_content[:120].strip()}...")
    print()

QUERY: How do I add a sub-task?
  [1] getting_started.txt: ## Creating and Assigning Tasks
1. Open a project and click "+ Add Task".
2. Fill in the task title, description, due da...
  [2] features.txt: ## Task Management
- Task statuses: To Do, In Progress, In Review, Done, Blocked.
- Sub-tasks: Supported on Pro and abov...
  [3] troubleshooting.txt: **Problem: Task not appearing after creation.**
Solution:
1. Refresh the page (F5 or Ctrl+R).
2. Check that you are view...

QUERY: What is the refund policy for annual subscriptions?
  [1] billing_policy.txt: ## Refund Policy
- Monthly subscriptions: No refunds are issued for partial months already paid.
- Annual subscriptions:...
  [2] billing_policy.txt: TASKFLOW PRO â€” BILLING & REFUND POLICY

## Subscription Plans and Pricing
- F...
  [3] billing_policy.txt: ## Cancelling Your Subscription
You can cancel your subscription at any time from Settings > Billing > Cancel Subscripti...

QUERY: Why is my Gantt chart loading slowly?
  [1] k

## 4.5 Build the RAG Chain

In [6]:
RAG_PROMPT_TEMPLATE = """You are the TaskFlow Pro Support Assistant.
Use ONLY the following retrieved documentation to answer the question.
If the answer is not in the context, say: "I don't have verified information on that.
I recommend contacting support@taskflowpro.com."
Never fabricate information not present in the context.

Context:
{context}

Question: {question}
Answer:"""

rag_prompt = PromptTemplate(
input_variables=["context", "question"],
template=RAG_PROMPT_TEMPLATE,
)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

#LCEL chain: retrieves docs once, returns both the answer and source documents
rag_chain_with_sources = (
RunnableParallel(
source_documents=retriever,
question=RunnablePassthrough()
).assign(
result=lambda x: (rag_prompt | llm | StrOutputParser()).invoke(
{
"context": format_docs(x["source_documents"]),
"question": x["question"],
}
)
)
)
print("RAG chain ready.")

RAG chain ready.


## 4.6 Compare: With vs Without Retrieval

In [7]:
from langchain_core.messages import SystemMessage, HumanMessage

NO_RAG_SYSTEM = "You are a support agent for TaskFlow Pro. Answer user questions helpfully."

comparison_queries = [
"What is the maximum file attachment size on the Free plan?",
"Is there a refund if I cancel my annual plan after 45 days?",
"Does TaskFlow support SSO?",
]

print("=" * 70)
for q in comparison_queries:
    # Without RAG
    no_rag_resp = llm.invoke(
    [SystemMessage(content=NO_RAG_SYSTEM), HumanMessage(content=q)]
    ).content.strip()
    # With RAG
    rag_result = rag_chain_with_sources.invoke(q)
    rag_resp = rag_result["result"].strip()
    sources = [os.path.basename(d.metadata.get("source", "")) for d in rag_result["source_documents"]]

    print(f"QUERY      : {q}")
    print(f"WITHOUT RAG: {no_rag_resp[:200]}")
    print(f"WITH RAG   : {rag_resp[:200]}")
    print(f"SOURCES    : {sources}")
    print("-" * 70) 

QUERY      : What is the maximum file attachment size on the Free plan?
WITHOUT RAG: On the Free plan of TaskFlow Pro, the maximum file attachment size is 10 MB per file. If you need to attach larger files, you might consider upgrading to a paid plan, which offers higher limits. Let m
WITH RAG   : The maximum file attachment size on the Free plan is 100 MB.
SOURCES    : ['troubleshooting.txt', 'features.txt', 'known_issues.txt']
----------------------------------------------------------------------
QUERY      : Is there a refund if I cancel my annual plan after 45 days?
WITHOUT RAG: TaskFlow Pro offers a 30-day money-back guarantee for annual plans. If you cancel your plan within the first 30 days, you are eligible for a full refund. However, since you are past the 30-day window,
WITH RAG   : I don't have verified information on that. I recommend contacting support@taskflowpro.com.
SOURCES    : ['billing_policy.txt', 'billing_policy.txt', 'billing_policy.txt']
-------------------------

## 4.7 Handle Missing Information

In [8]:
out_of_scope = [
"Does TaskFlow Pro offer an offline desktop app?",
"Can I import projects from Basecamp?",
]

print("Out-of-scope / missing information test:")
print("-" * 60)
for q in out_of_scope:
    result = rag_chain_with_sources.invoke(q)
    print(f"Q: {q}")
    print(f"A: {result['result'].strip()}")
    print()

Out-of-scope / missing information test:
------------------------------------------------------------
Q: Does TaskFlow Pro offer an offline desktop app?
A: I don't have verified information on that. I recommend contacting support@taskflowpro.com.

Q: Can I import projects from Basecamp?
A: I don't have verified information on that. I recommend contacting support@taskflowpro.com.



## 4.8 Summary

| Aspect | Without RAG | With RAG |
|---|---|---|
| Factual accuracy | Model may hallucinate specifics (file size, pricing thresholds) | Grounded in actual doc content |
| Source attribution | None | Returns source document name |
| Out-of-scope handling | Often invents a plausible answer | Returns "I don't have verified information" |
| Billing policy precision | Approximate | Exact (30-day annual refund window) |

**Conclusion:** RAG significantly reduces hallucination and allows precise, attributable answers. The retriever will be incorporated into the full agent in Phase 5.